In [ ]:
# | default_exp features.fsr

In [ ]:
# | export
from __future__ import annotations
import pandas as pd
import structlog
from kreview.eval_engine import FeatureEvaluator

log = structlog.get_logger()

In [ ]:
# | export
class FSROnTargetEvaluator(FeatureEvaluator):
    """Extracts the short/long fragment size ratio across on-target genomic bins.

    Only bins with read coverage (total_count > 0) are included in aggregation.
    On-target panels typically cover ~2% of bins; without this filter,
    the 98% zero-coverage bins dominate the median with zero values.
    """

    name = "FsrOnTarget"
    source_file = ".FSR.ontarget.parquet"
    tier = 1
    category = "fragmentation"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        extracted = {}
        try:
            if df.empty:
                return extracted

            # Filter to bins with actual read coverage
            if "total_count" in df.columns:
                n_total = len(df)
                df = df[df["total_count"] > 0]
                extracted["n_covered_bins"] = len(df)
                extracted["n_total_bins"] = n_total
                if df.empty:
                    log.warning(
                        "no_covered_bins",
                        evaluator=self.name,
                        total_bins=n_total,
                    )
                    return extracted

            cols = set(df.columns)
            target_metrics = [
                "short_long_ratio",
                "short_long_log2",
                "short_frac",
                "long_frac",
            ]

            for m in target_metrics:
                if m in cols:
                    extracted[f"global_{m}"] = float(df[m].median())

            return extracted
        except Exception as e:
            log.exception("extraction_failed", evaluator=self.name, error=str(e))
            return {}

In [ ]:
# | test
evaluator = FSREvaluator()
df_test = pd.DataFrame({"short_long_ratio": [1.5], "short_frac": [0.6]})
res = evaluator.extract(df_test)
assert res["global_short_long_ratio"] == 1.5
assert res["global_short_frac"] == 0.6